# 06 — Proposta: CatBoostClassifier (não executado)

Este notebook propõe um novo modelo que **não** foi usado em `05_train_baseline_model.ipynb`:
- Modelo principal: **CatBoostClassifier** (pode lidar bem com categorias de alta cardinalidade e interações).
- Alternativa/Extensão: Stacking ensemble (LightGBM + RandomForest + CatBoost) com meta-classifier `LogisticRegression`.

Objetivo: melhorar `precision` e `recall` em relação aos modelos já testados (LogisticRegression, DecisionTree, RandomForest, LightGBM).

**Observação:** este notebook contém somente código sugerido e instruções — NÃO executar aqui.

## Racional

- O notebook `05` já treina `LogisticRegression`, `DecisionTree`, `RandomForest` e `LightGBM`.
- `CatBoost` frequentemente tem bom desempenho em datasets mistos (numéricos + categorias) e pode exigir menos pré-processamento das categorias do que LightGBM/XGBoost.
- Também proponho testar *stacking* (combinar fortes modelos) e/ou técnicas de tratamento de desequilíbrio (focal loss, oversampling, `scale_pos_weight`) para melhorar *precision*/*recall*.

In [ ]:
# Se precisar instalar: pip install catboost scikit-learn pandas matplotlib seaborn
# !pip install catboost --quiet


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score
from catboost import CatBoostClassifier
from sklearn.ensemble import StackingClassifier, RandomForestClassifier
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
# Caminho compatível com o notebook anterior
INPUT_PATH = "../../data_lake/ml/datasets/df_features.parquet"
df = pd.read_parquet(INPUT_PATH)

# Features e target (mesma convenção do notebook 05)
X = df.drop(columns=["is_late_delivery"])
y = df["is_late_delivery"]

# Treino/teste
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


In [ ]:
# Observação sobre pré-processamento: se já houve one-hot encoding, CATBOOST receberá matrizes numéricas.
# Caso haja colunas categóricas originais (dtype 'object' / 'category'), prefira passar seus nomes para CatBoost via 'cat_features'.


In [ ]:
# Escalar colunas contínuas (mantendo abordagem do notebook 05)
continuous_cols = ["log_price", "log_freight", "log_weight", "log_volume", "freight_ratio", "distance_km"]
scaler = StandardScaler()
X_train[continuous_cols] = scaler.fit_transform(X_train[continuous_cols])
X_test[continuous_cols] = scaler.transform(X_test[continuous_cols])


In [ ]:
# ======= Proposta 1: CatBoostClassifier =======
# RAZÃO: bom manejo de categorias e regularização interna; histórico de bom desempenho em competições.

cat_params = {
    'iterations': 1000,
    'learning_rate': 0.05,
    'depth': 6,
    'loss_function': 'Logloss',
    'eval_metric': 'AUC',
    'random_seed': 42,
    'verbose': 100,
    'class_weights': None  # ajustar se desejar (ex: [1, scale_pos_weight])
}

cat = CatBoostClassifier(**cat_params)
# cat.fit(X_train, y_train, eval_set=(X_test, y_test), use_best_model=True)  # NÃO executar aqui


In [ ]:
# Código de avaliação por thresholds (mesma lógica do notebook 05)
# Depois de treinar, calcular y_prob = cat.predict_proba(X_test)[:,1] e testar thresholds
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7]

def evaluate_probs(y_test, y_prob, thresholds):
    results = []
    for t in thresholds:
        y_pred = (y_prob >= t).astype(int)
        precision = precision_score(y_test, y_pred)
        recall = recall_score(y_test, y_pred)
        f1 = f1_score(y_test, y_pred)
        auc = roc_auc_score(y_test, y_prob)
        results.append({
            'threshold': t, 'precision': precision, 'recall': recall, 'f1': f1, 'auc': auc
        })
    return pd.DataFrame(results)

# Exemplo (após treino):
# y_prob = cat.predict_proba(X_test)[:,1]
# evaluate_probs(y_test, y_prob, thresholds)


In [ ]:
# ======= Proposta 2: Stacking (opcional) =======
# Base learners: LightGBM (já testado), RandomForest, CatBoost; meta: LogisticRegression.

estimators = [
    ('rf', RandomForestClassifier(n_estimators=200, max_depth=12, class_weight='balanced', random_state=42)),
    ('cat', CatBoostClassifier(iterations=500, learning_rate=0.05, depth=6, random_seed=42, verbose=0))
]
stack = StackingClassifier(estimators=estimators, final_estimator=LogisticRegression(), n_jobs=-1, passthrough=False)
# stack.fit(X_train, y_train)  # NÃO executar aqui


## Notas de tuning e validação
- Usar `StratifiedKFold` para validação cruzada (k=3..5).
- Testar `class_weights`, `scale_pos_weight` (para XGBoost/LightGBM) ou oversampling (SMOTE) para melhorar recall sem degradar precision.
- Buscar `learning_rate` + `iterations` via `GridSearchCV` ou `BayesianOptimization`.
- Para stacking, avalie calibrar probabilidades do meta-classificador.

## Próximos passos (sugestões)
1. Executar o treinamento do `CatBoost` proposto com `eval_set` e salvar métricas por threshold.
2. Comparar precision/recall/f1/auc com os modelos já treinados no notebook `05`.
3. Se necessário, testar stacking e/ou técnicas de balanceamento.
4. Fazer tuning fino usando validação estratificada e otimização bayesiana.

Quer que eu: (A) rode experimentos e gere comparações automaticamente, ou (B) apenas deixe o notebook pronto para você executar?